# 03 — Explainability, Interpretability & Faithfulness Evaluation
**Explainable Multimodal Quantum-GNN Framework for Multi-Disease Chest X-ray Diagnosis**

This notebook implements and visualizes the complete explainability and faithfulness evaluation suite:
1. **Image Explainability (Grad-CAM++ & ViT)**: Disease-specific saliency heatmaps overlaid on original CXR scans for CNN and Vision Transformer backbones.
2. **Tabular Explainability (SHAP)**: Exact analytical Shapley feature attributions for clinical metadata (Age, Sex) across all 14 thoracic disease targets without target label leakage.
3. **Scan Graph Explainability (GNNExplainer)**: Longitudinal patient scan-graph edge and node importance identifying influential prior and concurrent scans.
4. **Disease-Specific Multimodal Fusion Attribution**: Exact 3-player Shapley attribution (Image vs Clinical Meta vs Scan Graph) quantifying modality reliance for each disease head.
5. **Quantitative Faithfulness Evaluation**: Pixel deletion/insertion curves, tabular feature perturbation tests, graph edge drop tests, and modality ablation benchmarks.

> **Important Clinical Disclaimer:** Modality and feature attributions reflect mathematical representation sensitivity within the trained network and do not establish clinical causality.

## Cell 1 — Environment Setup & Dependencies

In [ ]:
import subprocess, sys
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

pkgs = [
    'timm',
    'datasets',
    'scikit-learn',
    'pandas',
    'matplotlib',
    'PyYAML',
    'Pillow',
    'pennylane',
    'pennylane-lightning',
    'torch-geometric',
    'networkx',
    'scipy',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('All required dependencies are installed.')

## Cell 2 — Mount Google Drive / Clone Repository

In [ ]:
import os, sys

# ── Option A: Google Drive (if project folder is stored there) ─────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/urop'
    if os.path.exists(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Using repo from Drive: {REPO_PATH}')
    else:
        raise FileNotFoundError(f'Repo not found at {REPO_PATH}')
except Exception as e:
    print(f'Drive mount skipped ({e}). Working in local or cloned repo...')
    REPO_PATH = os.getcwd()

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print('Working Directory:', os.getcwd())

## Cell 3 — Load Model & Test Records

In [ ]:
import yaml
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

from src.models.model_factory import build_model
from src.dataset.label_encoder import DISEASE_NAMES
from src.graph.graph_builder import build_batch_scan_graph

with open('config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Disease Targets (14):', DISEASE_NAMES)

# Build default multimodal architecture (or load from checkpoint)
model_name = 'quantum_gnn_multimodal'
model = build_model(model_name, pretrained=False, device=device)
model.eval()
print(f'Loaded model: {model_name} on {device}')

# Prepare synthetic or loaded evaluation batch
B = 4
sample_images = torch.rand(B, 3, 224, 224).to(device)
sample_meta = torch.tensor([[0.25, 1.0], [-0.4, 0.0], [0.8, 1.0], [0.1, 0.0]], dtype=torch.float32).to(device)
patient_ids = ['P001', 'P001', 'P002', 'P003']
scan_ids = ['S101', 'S102', 'S201', 'S301']

gx, edge_index = build_batch_scan_graph(sample_meta.cpu(), patient_ids, add_self_loops=True)
gx = gx.to(device)
edge_index = edge_index.to(device)
print(f'Batch Graph: {gx.shape[0]} nodes, {edge_index.shape[1]} edges across {len(set(patient_ids))} patients')

## Cell 4 — Image Explainability (Grad-CAM++ & ViT Attribution)
Generates disease-specific spatial heatmaps and overlays for selected pathology targets (e.g. Cardiomegaly, Pneumothorax).

In [ ]:
import matplotlib.pyplot as plt
from src.explainability.gradcam import get_image_explainer
from src.explainability.visualizer import ExplainabilityVisualizer

img_explainer = get_image_explainer(model)

# Compare explanations for two different diseases on the same scan
target_diseases = [('Cardiomegaly', 1), ('Pneumothorax', 10)]
raw_img = (sample_images[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img)
axes[0].set_title('Original Chest X-ray', fontsize=12, fontweight='bold')
axes[0].axis('off')

for idx, (disease_name, class_idx) in enumerate(target_diseases, start=1):
    hmap = img_explainer.generate(
        sample_images[:1],
        sample_meta[:1],
        target_class=class_idx,
        graph_x=gx[:1],
        graph_edge_index=torch.zeros((2, 0), dtype=torch.long, device=device)
    )[0]
    overlay = ExplainabilityVisualizer.overlay_heatmap_on_image(raw_img, hmap, alpha=0.45)
    axes[idx].imshow(overlay)
    axes[idx].set_title(f'Grad-CAM++: {disease_name}', fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Cell 5 — Tabular Explainability (Exact Analytical Shapley Values)
Quantifies the impact of clinical metadata (`Age` and `Sex`) on predictions without target label leakage.

In [ ]:
from src.explainability.tabular_shap import TabularFeatureExplainer

tab_explainer = TabularFeatureExplainer(model, num_classes=14)
shap_results = tab_explainer.explain_instance(sample_images, sample_meta)

print('Shapley Values for Scan 0 (First 5 diseases):')
for d_idx in range(5):
    d_name = DISEASE_NAMES[d_idx]
    age_val = shap_results['Age'][0, d_idx]
    sex_val = shap_results['Sex'][0, d_idx]
    print(f'  {d_name:<16}: Age SHAP = {age_val:+.4f}, Sex SHAP = {sex_val:+.4f}')

fig = ExplainabilityVisualizer.plot_tabular_shap(
    shap_results,
    sample_idx=0,
    disease_name='Cardiomegaly',
    disease_idx=1,
    scan_id='S101'
)
plt.show()

## Cell 6 — Scan Graph Explainability (GNNExplainer)
Visualizes scan-to-scan connectivity and isolates influential neighboring scans for multi-scan patients.

In [ ]:
from src.explainability.gnn_explainer import ScanGraphExplainer

graph_explainer = ScanGraphExplainer(model, num_classes=14)
graph_exp = graph_explainer.explain(
    sample_images,
    sample_meta,
    gx,
    edge_index,
    target_class=1,
    patient_ids=patient_ids,
    scan_ids=scan_ids
)

print('Edge Importances:')
for u, v, w in zip(graph_exp['edge_index'][0], graph_exp['edge_index'][1], graph_exp['edge_importance']):
    print(f'  Scan {scan_ids[u]} -> Scan {scan_ids[v]}: Importance = {w:.4f}')

fig = graph_explainer.visualize_graph(graph_exp, title='Patient Scan-to-Scan Graph Explanation (Cardiomegaly)')
plt.show()

## Cell 7 — Multimodal Fusion Attribution (Image vs Clinical Meta vs Scan Graph)
Computes exact 3-player Shapley values ($2^3=8$ coalitions) measuring how much each modality drives predictions across all 14 diseases.

In [ ]:
from src.explainability.modality_attribution import MultimodalAttribution

mm_attr = MultimodalAttribution(model, num_classes=14)
attr_res = mm_attr.attribute_instance(sample_images, sample_meta, gx, edge_index)

mean_I = np.abs(attr_res['Image']).mean(axis=0)
mean_M = np.abs(attr_res['Clinical_Meta']).mean(axis=0)
mean_G = np.abs(attr_res['Graph']).mean(axis=0)
total = mean_I + mean_M + mean_G + 1e-8

pct_I = (mean_I / total) * 100
pct_M = (mean_M / total) * 100
pct_G = (mean_G / total) * 100

modality_df = pd.DataFrame({
    'Disease': DISEASE_NAMES,
    'Image (%)': np.round(pct_I, 2),
    'Clinical_Meta (%)': np.round(pct_M, 2),
    'Graph (%)': np.round(pct_G, 2),
})
print(modality_df.to_string(index=False))

fig = ExplainabilityVisualizer.plot_modality_contributions(
    pct_I, pct_M, pct_G, DISEASE_NAMES, title='Multimodal Attribution by Modality across 14 Diseases'
)
plt.show()

## Cell 8 — Quantitative Faithfulness Evaluation
Measures explanation validity via pixel deletion/insertion curves, tabular perturbations, and graph edge removal.

In [ ]:
from src.explainability.faithfulness import FaithfulnessEvaluator

evaluator = FaithfulnessEvaluator(model, img_explainer, tab_explainer, graph_explainer)
faith_metrics = evaluator.evaluate_batch(sample_images, sample_meta, gx, edge_index, target_class=1)

print('=== Faithfulness Evaluation Metrics ===')
print(f'Image Deletion AUC:     {faith_metrics["image_deletion_auc"]:.4f} (lower is better, confirms sensitive pixel removal)')
print(f'Image Insertion AUC:    {faith_metrics["image_insertion_auc"]:.4f} (higher is better, confirms salient pixel sufficiency)')
print(f'Tabular Perturbation:   {faith_metrics["tabular_perturbation_impact"]:.4f} (prediction drop on salient feature shuffle)')
print(f'Graph Edge Drop Impact: {faith_metrics["graph_edge_drop_impact"]:.4f} (prediction drop on important edge removal)')

del_steps = np.linspace(0.0, 1.0, len(faith_metrics['image_deletion_curve']))
ins_steps = np.linspace(0.0, 1.0, len(faith_metrics['image_insertion_curve']))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(del_steps, faith_metrics['image_deletion_curve'], 'r-o', label=f'Deletion Curve (AUC={faith_metrics["image_deletion_auc"]:.3f})')
ax.plot(ins_steps, faith_metrics['image_insertion_curve'], 'g-s', label=f'Insertion Curve (AUC={faith_metrics["image_insertion_auc"]:.3f})')
ax.set_xlabel('Fraction of Pixels Modified', fontsize=11)
ax.set_ylabel('Model Target Logit', fontsize=11)
ax.set_title('Faithfulness: Image Deletion vs Insertion Curves', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()